# p3 · Concept(=Object Type) 추출

p2의 normed_term을 묶어 **concept = Object Type** 초안을 만든다. term은 인스턴스, concept은 타입이다.

각 concept에 normed_term을 세 역할로 붙인다.
- `evidence_term_ids`: 이 타입 자체를 가리키는 표기 (예: `purchase_order_item`)
- `instance_term_ids`: 이 타입의 개별 값·인스턴스 (예: `two_way_matching` → ItemCategory)
- `property_term_ids`: 이 타입의 속성 (예: `invoice_value` → Invoice)

**Gate 3**
1. instance가 concept로 올라가지 않았는가 → 실패 시 p2로
2. 기존 Object Type 재사용이 먼저 (닫힌 어휘). 카탈로그 밖 `reuse_of`, 의미 중복 → p3로
3. 계층의 순환·방향 검사 → 문제 간선만 제거, p3로
4. concept마다 근거 term 최소 개수 → 부족하면 p1로

검사 순서는 싼 것부터: L1 결정론 → L3 LLM judge(독립 호출, 새 컨텍스트, concept 한 건씩) → 낮은 confidence는 사람 큐.

참고: AutoSchemaKG(bottom-up 일반화), OntoGen(상위 category 먼저), OntoEKG(클래스 자리에 개체 생성·상호 subclass 오류 = 반례).

입력: `outputs/p2/normed_terms.json` (+ 선택: `catalog/object_types.json`) → 출력: `outputs/p3/`

## 설정

In [1]:
import re
from collections import Counter, defaultdict
from typing import Literal

from pydantic import BaseModel, Field

from common import GEMINI_MODEL, MODULE_DIR, OUTPUT_ROOT, gate_record, generate_json, read_json, stage_dir, write_json

P2 = read_json(OUTPUT_ROOT / "p2" / "normed_terms.json")
OUTPUT_DIR = stage_dir("p3")

# 기존 Object Type 카탈로그 (닫힌 어휘). [{"name": ..., "description": ...}]. 없으면 빈 목록.
CATALOG_PATH = MODULE_DIR / "catalog" / "object_types.json"
CATALOG = read_json(CATALOG_PATH) if CATALOG_PATH.exists() else []
CATALOG_NAMES = {c["name"] for c in CATALOG}

MIN_EVIDENCE_MENTIONS = 2   # evidence + instance term의 원문 언급 합계 하한
JUDGE_CONF = 0.7            # judge confidence가 이보다 낮으면 사람 큐

normed = {n["normed_id"]: n for n in P2["normed_terms"]}
print(GEMINI_MODEL, "normed_terms:", len(normed), "catalog:", len(CATALOG))

gemini-3.5-flash normed_terms: 129 catalog: 0


## 제안: normed_term → concept 초안

한 번의 호출로 전체 normed_term 목록을 보고 concept을 제안한다. 멤버는 반드시 주어진 `normed_id`로만 적게 한다(닫힌 어휘).

In [2]:
class ConceptProposal(BaseModel):
    name: str = Field(description="Singular PascalCase type name, e.g. PurchaseOrderItem.")
    description: str = Field(description="One sentence definition grounded in the document.")
    reuse_of: str | None = Field(default=None, description="Name of an existing Object Type from the catalog if this is the same type; else null.")
    parent: str | None = Field(default=None, description="Name of a broader proposed concept or catalog type if this is a subtype; else null.")
    evidence_term_ids: list[str] = Field(description="normed_ids that name this type itself.")
    instance_term_ids: list[str] = Field(default_factory=list, description="normed_ids that are individual values/instances of this type.")
    property_term_ids: list[str] = Field(default_factory=list, description="normed_ids that are attributes of this type.")


class ConceptResult(BaseModel):
    concepts: list[ConceptProposal]


PROPOSE_SYSTEM = """You build the type layer (Object Types) of an ontology from normalized terms of one business process document.

Terms are instances; concepts are types. For each concept:
- Name it as a singular PascalCase type.
- Attach terms only by their given normed_id, in exactly one role: evidence (names the type itself), instance (an individual value of the type, e.g. a specific category value or a named variant), or property (an attribute of the type).
- A named value such as a specific category, variant, or status is an instance, never its own concept, unless the document treats it as a type with its own instances.
- Business events and activities that are recorded with their own attributes may be concepts (e.g. GoodsReceipt).
- Prefer reusing a catalog type (set reuse_of) over creating a new one.
- Use parent only for a true subtype (is-a) relation, never for part-of or has.
- Skip generic or meta terms that do not describe the business domain (e.g. about the report or analysis itself).
Every concept needs at least one evidence or instance term."""


def term_line(n: dict) -> str:
    sent = n["mentions"][0]["sentence"]
    variants = "; ".join(n["surface_variants"])
    return f"- {n['normed_id']} | kind={n['kind']} | variants: {variants} | e.g. \"{sent}\""


catalog_block = "\n".join(f"- {c['name']}: {c.get('description', '')}" for c in CATALOG) or "(empty)"
propose_prompt = (
    f"Existing Object Type catalog:\n{catalog_block}\n\n"
    f"Normalized terms (normed_id | kind hint | surface variants | example sentence):\n"
    + "\n".join(term_line(n) for n in normed.values())
    + "\n\nPropose the concepts."
)

proposal, err = generate_json(propose_prompt, ConceptResult, PROPOSE_SYSTEM)
assert proposal is not None, err
proposals = proposal.concepts
print("proposed concepts:", len(proposals))
for c in proposals:
    print(f"  {c.name:<28} parent={c.parent}  ev={len(c.evidence_term_ids)} inst={len(c.instance_term_ids)} prop={len(c.property_term_ids)}")

proposed concepts: 22
  Company                      parent=None  ev=4 inst=0 prop=1
  Vendor                       parent=None  ev=1 inst=0 prop=1
  PurchaseOrder                parent=None  ev=1 inst=6 prop=2
  PurchaseOrderItem            parent=None  ev=3 inst=0 prop=5
  PurchaseRequisition          parent=None  ev=1 inst=0 prop=0
  OrderConfirmation            parent=None  ev=1 inst=0 prop=0
  ServiceEntrySheet            parent=None  ev=1 inst=0 prop=0
  GoodsReceipt                 parent=None  ev=3 inst=0 prop=1
  Invoice                      parent=None  ev=7 inst=0 prop=1
  VendorDebitMemo              parent=None  ev=1 inst=0 prop=0
  Warehouse                    parent=None  ev=1 inst=0 prop=0
  Goods                        parent=None  ev=1 inst=0 prop=0
  Resource                     parent=None  ev=2 inst=2 prop=0
  InternalDepartment           parent=None  ev=1 inst=0 prop=0
  EventLog                     parent=None  ev=3 inst=0 prop=0
  Case                         pa

## Gate 3 · L1 결정론 검사

- 멤버 id가 p2의 normed_id인가 (닫힌 어휘, 아니면 그 id만 버리고 기록)
- 같은 이름 concept은 하나로 합친다
- `reuse_of`가 카탈로그에 있는가 → 없으면 p3로
- ① instance 승격: concept 이름의 key가 다른 concept의 instance term이거나, 근거 term이 전부 `identifier`면 p2로
- ④ 근거 언급 수 ≥ `MIN_EVIDENCE_MENTIONS` → 아니면 p1로

In [3]:
def to_key(name: str) -> str:
    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower().replace("__", "_")


def pascal(name: str) -> str:
    parts = re.split(r"[^A-Za-z0-9]+", name)
    return "".join(p[:1].upper() + p[1:] for p in parts if p)


rejected, dropped_members, merge_notes = [], [], []

# 같은 이름 합치기 + 닫힌 어휘 검사
by_name: dict[str, dict] = {}
for c in proposals:
    name = pascal(c.name)
    roles = {}
    for role in ("evidence_term_ids", "instance_term_ids", "property_term_ids"):
        ids = []
        for i in getattr(c, role):
            if i in normed:
                ids.append(i)
            else:
                dropped_members.append({"concept": name, "role": role, "id": i, "reason": "unknown_normed_id"})
        roles[role] = ids
    if name in by_name:
        merge_notes.append({"name": name, "reason": "duplicate_name_merged"})
        for role, ids in roles.items():
            by_name[name][role] = sorted(set(by_name[name][role]) | set(ids))
        continue
    by_name[name] = {"name": name, "description": c.description, "reuse_of": c.reuse_of, "parent": pascal(c.parent) if c.parent else None, **roles}

instance_ids = {i for c in by_name.values() for i in c["instance_term_ids"]}


def evidence_mentions(c: dict) -> int:
    return sum(normed[i]["n_mentions"] for i in set(c["evidence_term_ids"]) | set(c["instance_term_ids"]))


l1_passed = []
for c in by_name.values():
    name = c["name"]
    ev = c["evidence_term_ids"]
    if c["reuse_of"] and c["reuse_of"] not in CATALOG_NAMES:
        rejected.append(gate_record(name, "p3", "reuse_of_not_in_catalog", "p3", reuse_of=c["reuse_of"]))
        continue
    if to_key(name) in instance_ids or (ev and all(normed[i]["kind"] == "identifier" for i in ev)):
        rejected.append(gate_record(name, "p3", "instance_promoted_to_concept", "p2", evidence_term_ids=ev))
        continue
    n_ev = evidence_mentions(c)
    if not (ev or c["instance_term_ids"]) or n_ev < MIN_EVIDENCE_MENTIONS:
        rejected.append(gate_record(name, "p3", "insufficient_evidence", "p1", evidence_mentions=n_ev))
        continue
    c["evidence_mentions"] = n_ev
    l1_passed.append(c)

print("L1 passed:", len(l1_passed), "| rejected:", Counter(r["reason"] for r in rejected), "| dropped members:", len(dropped_members))

L1 passed: 18 | rejected: Counter({'insufficient_evidence': 4}) | dropped members: 0


## Gate 3 · L3 LLM judge (concept 한 건씩, 새 컨텍스트)

제안 호출과 독립된 호출로 판정한다.
- `level`: `type`(타입) / `instance`(개별 값이 타입으로 올라감) / `not_a_type`(속성·상태·메타 용어)
- `duplicate_of`: 다른 concept과 같은 타입이면 그 이름

근거 언급이 많은 concept부터 판정하고, 이미 통과한 concept과 중복이라는 판정만 반영한다(서로를 중복으로 지목해 둘 다 빠지는 것을 막는다).

In [4]:
class ConceptJudgement(BaseModel):
    level: Literal["type", "instance", "not_a_type"]
    duplicate_of: str | None = Field(default=None, description="Name of another listed concept that is the same type, else null.")
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


JUDGE_SYSTEM = """You review one proposed ontology class (Object Type) for a business process ontology.
level=type if it is a class whose members are many individual things, events, documents, or roles.
level=instance if it is really one individual or one named value of some other class (e.g. a specific category value, a specific variant).
level=not_a_type if it is an attribute, a measurement, a state, or a term about the analysis/report rather than the domain.
duplicate_of: name another listed concept only if both denote the same class (synonyms), not if one is a subtype of the other."""


def judge_concept(c: dict, others: list[dict]) -> tuple[ConceptJudgement | None, str | None]:
    def labels(ids):
        return ", ".join(normed[i]["label"] for i in ids) or "-"
    sents = []
    for i in c["evidence_term_ids"] + c["instance_term_ids"]:
        s = normed[i]["mentions"][0]["sentence"]
        if s not in sents:
            sents.append(s)
    prompt = (
        f"Proposed concept: {c['name']}\nDefinition: {c['description']}\n"
        f"Names it: {labels(c['evidence_term_ids'])}\nInstances: {labels(c['instance_term_ids'])}\nProperties: {labels(c['property_term_ids'])}\n"
        f"Evidence sentences:\n" + "\n".join(f"  - {s}" for s in sents[:4]) + "\n\n"
        f"Other concepts:\n" + "\n".join(f"- {o['name']}: {o['description']}" for o in others) + "\n\nJudge the proposed concept."
    )
    return generate_json(prompt, ConceptJudgement, JUDGE_SYSTEM)


passed, human_queue = [], []
for c in sorted(l1_passed, key=lambda c: -c["evidence_mentions"]):
    j, err = judge_concept(c, [o for o in l1_passed if o is not c])
    if j is None:
        human_queue.append(gate_record(c["name"], "p3", "judge_failed", "human", error=err))
        continue
    c["judge"] = j.model_dump()
    if j.level == "instance":
        rejected.append(gate_record(c["name"], "p3", "instance_promoted_to_concept", "p2", judge=c["judge"]))
    elif j.level == "not_a_type":
        rejected.append(gate_record(c["name"], "p3", "not_a_type", "p3", judge=c["judge"]))
    elif j.duplicate_of and j.duplicate_of in {p["name"] for p in passed}:
        rejected.append(gate_record(c["name"], "p3", "duplicate_concept", "p3", duplicate_of=j.duplicate_of, judge=c["judge"]))
    elif j.confidence < JUDGE_CONF:
        human_queue.append(gate_record(c["name"], "p3", "low_confidence", "human", judge=c["judge"]))
    else:
        passed.append(c)

print("passed:", len(passed), "| rejected:", Counter(r["reason"] for r in rejected), "| human:", len(human_queue))
for r in rejected:
    print(f"  REJECT {r['item_id']:<28} {r['reason']:<30} -> {r['route_to']}")

passed: 17 | rejected: Counter({'insufficient_evidence': 4, 'not_a_type': 1}) | human: 0
  REJECT ServiceEntrySheet            insufficient_evidence          -> p1
  REJECT VendorDebitMemo              insufficient_evidence          -> p1
  REJECT Warehouse                    insufficient_evidence          -> p1
  REJECT InternalDepartment           insufficient_evidence          -> p1
  REJECT PaymentBlock                 not_a_type                     -> p3


## Gate 3 · ③ 계층 검사

- 부모는 통과한 concept이나 카탈로그 타입이어야 한다. 아니면 간선만 제거
- 자기 자신이 부모인 경우, 순환이 생기는 경우 간선 제거 (OntoEKG의 상호 subclass 오류)

In [5]:
hierarchy_issues = []
valid_parents = {c["name"] for c in passed} | CATALOG_NAMES

for c in passed:
    p = c["parent"]
    if p and (p == c["name"] or p not in valid_parents):
        hierarchy_issues.append(gate_record(f"{c['name']}->{p}", "p3", "parent_self" if p == c["name"] else "parent_not_a_concept", "p3"))
        c["parent"] = None

parent_of = {c["name"]: c["parent"] for c in passed}
for c in passed:
    seen, cur = [], c["name"]
    while cur is not None and cur not in seen:
        seen.append(cur)
        cur = parent_of.get(cur)
    if cur is not None:   # 순환: 되돌아온 지점의 간선을 끊는다
        hierarchy_issues.append(gate_record(f"{cur}->{parent_of[cur]}", "p3", "hierarchy_cycle", "p3", cycle=seen[seen.index(cur):]))
        parent_of[cur] = None

for c in passed:
    c["parent"] = parent_of[c["name"]]

print("hierarchy issues:", [(h["item_id"], h["reason"]) for h in hierarchy_issues])

hierarchy issues: []


## 결과 확인과 저장

In [6]:
concepts = []
for c in sorted(passed, key=lambda c: c["name"]):
    member_ids = set(c["evidence_term_ids"]) | set(c["instance_term_ids"])
    concepts.append({
        "concept_id": c["name"],
        **{k: c[k] for k in ("name", "description", "reuse_of", "parent", "evidence_term_ids", "instance_term_ids", "property_term_ids", "evidence_mentions")},
        "surface_forms": sorted({v for i in c["evidence_term_ids"] for v in normed[i]["surface_variants"]}),
        "chunk_ids": sorted({ch for i in member_ids for ch in normed[i]["chunk_ids"]}),
        "judge": c.get("judge"),
    })

assigned = {i for c in concepts for r in ("evidence_term_ids", "instance_term_ids", "property_term_ids") for i in c[r]}
unassigned = sorted(set(normed) - assigned)

for c in concepts:
    inst = [normed[i]["label"] for i in c["instance_term_ids"]]
    print(f"{c['name']:<28} parent={str(c['parent']):<18} {c['description'][:70]}" + (f"\n{'':<30}instances: {inst}" if inst else ""))
print(f"\nnormed_terms assigned: {len(assigned)}/{len(normed)}")

BusinessRule                 parent=None               A policy, constraint, or matching procedure governing the purchasing p
                              instances: ['invoice-after-goods-receipt three-way matching', 'invoice-before-goods-receipt three-way matching']
Case                         parent=None               The organizational unit used to group events in the event log, typical
Company                      parent=None               A business entity or subsidiary operating within the paints and coatin
EventLog                     parent=None               A collection of recorded events used to analyze business processes.
Goods                        parent=None               Physical items or materials delivered to and stored in a warehouse.
GoodsReceipt                 parent=None               An activity or event recording the physical receipt of goods or servic
Invoice                      parent=None               A document or recorded event representing a request 

In [7]:
meta = {
    "model": GEMINI_MODEL,
    "catalog_size": len(CATALOG),
    "min_evidence_mentions": MIN_EVIDENCE_MENTIONS,
    "judge_conf": JUDGE_CONF,
    "n_normed_terms_in": len(normed),
    "n_proposed": len(proposals),
    "n_concepts": len(concepts),
    "n_rejected": len(rejected),
    "n_human_queue": len(human_queue),
    "n_hierarchy_issues": len(hierarchy_issues),
    "n_unassigned_terms": len(unassigned),
}
write_json(OUTPUT_DIR / "concepts.json", {"meta": meta, "concepts": concepts})
write_json(OUTPUT_DIR / "rejected.json", {"rejected": rejected, "hierarchy_issues": hierarchy_issues, "dropped_members": dropped_members, "merge_notes": merge_notes})
write_json(OUTPUT_DIR / "human_queue.json", {"human_queue": human_queue})
write_json(OUTPUT_DIR / "proposals.json", {"proposals": [p.model_dump() for p in proposals], "unassigned_terms": unassigned})
meta

{'model': 'gemini-3.5-flash',
 'catalog_size': 0,
 'min_evidence_mentions': 2,
 'judge_conf': 0.7,
 'n_normed_terms_in': 129,
 'n_proposed': 22,
 'n_concepts': 17,
 'n_rejected': 5,
 'n_human_queue': 0,
 'n_hierarchy_issues': 0,
 'n_unassigned_terms': 23}